In [ ]:
"""
Sweep the Kerr strength chi and plot, for each chi, the period-averaged
Hellinger distance between

  * Q_HB3   : the THIRD-CUMULANT (Edgeworth-corrected Gaussian) Husimi
              function rebuilt from the harmonic-balance moment solution
              of hhb/kerr/hb_moments_kerr_3.py (9 real DOFs incl. kappa_30, kappa_21),
  * Q_exact : the Husimi function of the truncated-Fock Lindblad steady state
              obtained with dynamiqs.

Q_HB3 is the Edgeworth expansion of the density whose cumulant-generating
function the closure truncates:
    Q_3(z) = Q_G(z) * [1 + 2 Re( kappa30/6 * h30(z) + kappa21/2 * h21(z) )]
with h_jk = (-d/dZ)^j (-d/dZ*)^k Q_G / Q_G (Wirtinger derivatives, Z = z-mu).
The correction integrates to zero and leaves the 2nd-order cumulants intact;
its integrated 3rd moments reproduce kappa_30, kappa_21 exactly (verified
numerically at 1e-13). Edgeworth tails can dip slightly below zero -- the
Hellinger integrand clips at 0, and Q_3 is clipped likewise.

Run:  python3 sweep_kerr_hellinger_3.py
"""

import numpy as np
import matplotlib.pyplot as plt

# Artefact locations resolve from the installed package, not the working
# directory: data/kerr/*.npz is the tracked regression baseline.
from hhb.paths import KERR_DATA_DIR as DATA_DIR
DATA_DIR.mkdir(parents=True, exist_ok=True)

from hhb.kerr.hb_moments_kerr_3 import KerrParams, HBSettingsMoments3, MomentHillMethod3

import dynamiqs as dq
dq.set_precision('double')
import jax.numpy as jnp


# ----------------------------------------------------------------------
# Fixed physical parameters (notebook values); chi is the swept quantity
# ----------------------------------------------------------------------
Delta = 0.1 *2*np.pi
kappa = .05*2*np.pi
eps = .1*2*np.pi
omega_d = 5*2*np.pi

CHI_VALUES = np.linspace(0.0, .1, 10)*2*np.pi

# HB settings
N_H = 14
SAMPLES_PER_HARMONIC = 48

# Fock / dynamiqs settings
N_FOCK = 30
N_PERIODS_TOTAL = 800        # transient + steady state
N_PERIODS_USE = 2           # periods kept for the phase average
N_SAVE = 4000

# phase-space grid for the Hellinger integral
GRID_LIM = 10.0
GRID_PTS = 100


# ----------------------------------------------------------------------
# Husimi functions
# ----------------------------------------------------------------------
def Q_hb_grid(X, Y, mu, sig, sigt):
    """Gaussian Q function; sig = sigma^2 = <da da^dag> (anti-normal), sigt = <da^2>."""
    Z = (X + 1j * Y) - mu
    denom = sig**2 - np.abs(sigt)**2
    return (1 / np.pi / np.sqrt(denom)
            * np.exp((-sig * np.abs(Z)**2 + np.real(np.conj(sigt) * Z**2)) / denom))


def Q_hb_grid_3(X, Y, mu, sig, sigt, k30, k21):
    """Edgeworth-corrected Gaussian Q carrying the third cumulants.

    Q_3 = Q_G * [1 + 2 Re(k30/6 * h30 + k21/2 * h21)],
    h_jk = (-d/dZ)^j (-d/dZ*)^k Q_G / Q_G.  With phi = log Q_G + const:
        g  = dphi/dZ  = (sigt* Z - sig Z*)/D,   gb = conj(g),
        A  = dg/dZ    = sigt*/D,                B  = dg/dZ* = -sig/D,
        h30 = -(g^3 + 3 g A),   h21 = -(g^2 gb + A gb + 2 g B).
    Clipped at 0 (Edgeworth tails can dip slightly negative).
    """
    Z = (X + 1j * Y) - mu
    D = sig**2 - np.abs(sigt)**2
    QG = (1 / np.pi / np.sqrt(D)
          * np.exp((-sig * np.abs(Z)**2 + np.real(np.conj(sigt) * Z**2)) / D))
    g = (np.conj(sigt) * Z - sig * np.conj(Z)) / D
    gb = np.conj(g)
    A = np.conj(sigt) / D
    B = -sig / D
    h30 = -(g**3 + 3 * g * A)
    h21 = -(g**2 * gb + A * gb + 2 * g * B)
    corr = 1 + 2 * np.real(k30 / 6 * h30 + k21 / 2 * h21)
    return np.clip(QG * corr, 0.0, None)


def coherent_overlaps(alpha_grid, n_fock):
    n = np.arange(n_fock)
    log_norm = -0.5 * np.abs(alpha_grid)[:, None]**2 - 0.5 * np.array(
        [np.sum(np.log(np.arange(1, k + 1))) for k in n])[None, :]
    return np.exp(log_norm) * alpha_grid[:, None]**n[None, :]   # (M, n_fock)


def make_Q_exact(C, shape):
    def Q_exact_grid(rho):
        rho = np.asarray(rho)
        Q_flat = np.real(np.einsum('mi,ij,mj->m', C.conj(), rho, C)) / np.pi
        return Q_flat.reshape(shape)
    return Q_exact_grid


def hellinger(Q1, Q2, dA):
    return np.sqrt(max(0.0, 1 - np.sum(np.sqrt(np.clip(Q1 * Q2, 0, None))) * dA))


def make_fourier_interp(samples, T):
    """samples: 1D array (real or complex) uniformly sampled on [0, T)."""
    N = samples.size
    coeffs = np.fft.fft(samples) / N
    freqs = np.fft.fftfreq(N, d=T / N) * 2 * np.pi
    def f_at(t):
        t = np.atleast_1d(t)
        return np.exp(1j * np.outer(t, freqs)) @ coeffs
    return f_at


# ----------------------------------------------------------------------
# The two solvers, one chi at a time
# ----------------------------------------------------------------------
def solve_hb(chi, z_guess=None):
    kerr = KerrParams(Delta=Delta, chi=chi, kappa=kappa, eps=eps, omega_d=omega_d)
    settings = HBSettingsMoments3(N_H=N_H, samples_per_harmonic=SAMPLES_PER_HARMONIC)
    hb = MomentHillMethod3(kerr, settings)

    if z_guess is None:
        z0 = np.zeros(hb.dim)
        z0[2] = np.sqrt(2) * 1.0    # DC sigma^2 guess (vacuum)
    else:
        z0 = z_guess                # numerical continuation in chi

    z_sol = hb.solve(z0)
    return hb, z_sol


def solve_exact(chi):
    """Lindblad evolution; returns (T, t_use, states_use)."""
    a = dq.destroy(N_FOCK)
    H_0 = Delta * a.dag() @ a + chi / 2 * (a.dag() @ a.dag()) @ (a @ a)
    H_drive_a = dq.modulated(lambda t: eps / 2 * (1 + jnp.exp(-2j * omega_d * t)), a)
    H_drive_a_dag = dq.modulated(lambda t: eps / 2 * (1 + jnp.exp(2j * omega_d * t)), a.dag())
    H_tot = H_0 + H_drive_a + H_drive_a_dag
    jump_op = [jnp.sqrt(kappa) * a]

    T = 2 * np.pi / (2 * omega_d)     # folding period of the drive
    t_max = N_PERIODS_TOTAL * T
    t_save = np.linspace(0, t_max, N_SAVE)

    psi_0 = dq.coherent(N_FOCK, -2)
    method = dq.method.Tsit5(atol=1e-12, rtol=1e-12)
    res = dq.mesolve(H_tot, jump_op, psi_0, t_save, method=method)

    mask = t_save >= t_max - N_PERIODS_USE * T
    idx = np.where(mask)[0]
    return T, t_save[idx], [res.states[i] for i in idx]


def mean_hellinger_for_chi(chi, X, Y, Q_exact_grid, dA, z_guess=None):
    hb, z_sol = solve_hb(chi, z_guess)
    X_hb = hb.x_tilde_to_X(hb.Gamma @ z_sol)      # (9, N): all third-order DOFs
    mu_hb = X_hb[0] + 1j * X_hb[1]
    sig_hb = X_hb[2]
    sigt_hb = X_hb[3] + 1j * X_hb[4]
    k30_hb = X_hb[5] + 1j * X_hb[6]
    k21_hb = X_hb[7] + 1j * X_hb[8]

    T, t_use, states_use = solve_exact(chi)

    # HB interpolants live on hb.T_fund; evaluating at absolute times is safe.
    mu_t = make_fourier_interp(mu_hb, hb.T_fund)(t_use)
    sig_t = np.real(make_fourier_interp(sig_hb, hb.T_fund)(t_use))
    sigt_t = make_fourier_interp(sigt_hb, hb.T_fund)(t_use)
    k30_t = make_fourier_interp(k30_hb, hb.T_fund)(t_use)
    k21_t = make_fourier_interp(k21_hb, hb.T_fund)(t_use)

    # Gaussian-part physicality: with n = <da+ da> = sigma^2 - 1 and m = <da^2>,
    # a single-mode Gaussian is physical iff n(n+1) >= |m|^2. (sigma^2 >= 1 + |m|
    # is too strict -- it flags the exact Lindblad state as unphysical.)
    n_occ = sig_t - 1.0
    viol = n_occ * (n_occ + 1.0) < np.abs(sigt_t)**2 - 1e-9
    n_viol = int(viol.sum())

    distances = np.array([
        hellinger(Q_exact_grid(rho_k),
                  Q_hb_grid_3(X, Y, mu_t[k], sig_t[k], sigt_t[k], k30_t[k], k21_t[k]),
                  dA)
        for k, rho_k in enumerate(states_use)
    ])

    # fold onto one period and average with the trapezoid rule
    t_mod = t_use % T
    order = np.argsort(t_mod)
    avg = np.trapezoid(distances[order], t_mod[order]) / T

    return avg, distances.max(), n_viol, len(distances), z_sol

In [ ]:
real_axis = np.linspace(-GRID_LIM, GRID_LIM, GRID_PTS)
imag_axis = np.linspace(-GRID_LIM, GRID_LIM, GRID_PTS)
X, Y = np.meshgrid(real_axis, imag_axis)
dA = (real_axis[1] - real_axis[0]) * (imag_axis[1] - imag_axis[0])

C = coherent_overlaps((X + 1j * Y).ravel(), N_FOCK)
Q_exact_grid = make_Q_exact(C, X.shape)

avg_list, max_list = [], []
z_guess = None

for chi in CHI_VALUES:
    avg, dmax, n_viol, n_pts, z_guess = mean_hellinger_for_chi(
        chi, X, Y, Q_exact_grid, dA, z_guess
    )
    avg_list.append(avg)
    max_list.append(dmax)
    note = f"  [unphysical at {n_viol}/{n_pts} phases]" if n_viol else ""
    print(f"chi = {chi:.4f}   mean Hellinger = {avg:.4f}   max = {dmax:.4f}{note}")

avg_arr = np.array(avg_list)
max_arr = np.array(max_list)


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(CHI_VALUES, avg_arr, 'o-', label='period-averaged (3rd order)')
plt.plot(CHI_VALUES, max_arr, 's--', color='C0', alpha=0.7, label='worst phase')
try:
    ref = np.load(DATA_DIR / 'sweep_kerr_hellinger_realistic.npz')
    plt.plot(ref['chi'], ref['mean'], 'o-', color='C3', alpha=0.6,
             label='period-averaged (Gaussian)')
    plt.plot(ref['chi'], ref['max'], 's--', color='C3', alpha=0.6,
            label=f'max (Gaussian)')
except FileNotFoundError:
    pass
plt.xlabel(r'Kerr strength $\chi$')
plt.ylabel('Hellinger distance')
plt.title(r'$Q_{\rm HB3}$ vs $Q_{\rm exact}$, averaged over one drive period')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
np.savez(DATA_DIR / 'sweep_kerr_hellinger_3_realistic.npz',
         chi=CHI_VALUES, mean=avg_arr, max=max_arr)